[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/34_representation_learning/02_variational_autoencoder.ipynb)

# Bài 02 — Variational Autoencoder (VAE)

**🎯 Mục tiêu**
- Vì sao autoencoder thường **không sinh được** dữ liệu mới.
- VAE: encoder xác suất, **reparameterization trick**, **ELBO** = tái tạo − KL; cài và train trên ảnh hình học.
- Sinh mẫu từ prior, **nội suy** trong không gian latent.
- Hệ số $\beta$: đánh đổi tái tạo ↔ cấu trúc latent (rate–distortion).
- Đánh giá mô hình sinh khó thế nào; bản đồ các họ mô hình sinh (GAN, diffusion, autoregressive).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/34_representation_learning

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import SHAPE_CLASSES, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
to_tensor = lambda X: torch.tensor(X[:, None], dtype=torch.float32)
X_train, y_train = make_shapes(20000, seed=0, noise=0.05)
X_test, y_test = make_shapes(2000, seed=1, noise=0.05)
X_train_t, X_test_t = to_tensor(X_train), to_tensor(X_test)
print("thiết bị:", device)


def show_grid(images, title, ncols=12):
    fig, axes = plt.subplots(1, ncols, figsize=(13, 1.4))
    for ax, img in zip(axes, images):
        ax.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    fig.suptitle(title, fontsize=10, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()

## 1. Autoencoder không phải mô hình sinh

Một mô hình **sinh** cho phép lấy mẫu $x \sim p(x)$ — tạo ảnh mới giống dữ liệu. Ý tưởng ngây thơ: lấy $z$ ngẫu nhiên rồi đưa qua decoder của autoencoder (bài 01). Vấn đề: không có gì ràng buộc **latent của dữ liệu nằm ở đâu** — các mã $z$ của ảnh thật rải rác trong một vùng nhỏ, hình dạng bất kỳ; điểm ngẫu nhiên gần như chắc chắn rơi vào vùng decoder chưa từng thấy.

Mô hình dưới đây dùng chung kiến trúc với VAE (mục 2), chỉ khác: train như **autoencoder thường** (bỏ phần ngẫu nhiên và KL).

In [ ]:
K = 16


class VAE(nn.Module):
    def __init__(self, k=K):
        super().__init__()
        self.k = k
        self.encoder = nn.Sequential(nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
                                     nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
                                     nn.Flatten(), nn.Linear(64 * 7 * 7, 2 * k))            # → (μ, log σ²)
        self.decoder = nn.Sequential(nn.Linear(k, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
                                     nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
                                     nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1))     # logits của pixel

    def encode(self, x):
        mu, logvar = self.encoder(x).chunk(2, dim=1)
        return mu, logvar

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)          # reparameterization trick
        return self.decoder(z), mu, logvar

    @torch.no_grad()
    def sample(self, n, generator):
        return torch.sigmoid(self.decoder(torch.randn(n, self.k, generator=generator).to(device))).cpu()


def elbo_terms(logits, x, mu, logvar):
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / len(x)          # −log p(x|z), mỗi ảnh
    kl = (-0.5 * (1 + logvar - mu**2 - logvar.exp()).sum(1)).mean()                         # KL(q(z|x) ‖ N(0, I))
    return recon, kl


def train(beta, steps=5000, plain_autoencoder=False, seed=0):
    torch.manual_seed(seed)
    model = VAE().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        xb = X_train_t[torch.randint(0, len(X_train_t), (128,), generator=g)].to(device)
        opt.zero_grad()
        if plain_autoencoder:
            mu, _ = model.encode(xb)
            loss = F.binary_cross_entropy_with_logits(model.decoder(mu), xb, reduction="sum") / len(xb)
        else:
            logits, mu, logvar = model(xb)
            recon, kl = elbo_terms(logits, xb, mu, logvar)
            loss = recon + beta * kl
        loss.backward()
        opt.step()
    return model.eval()


t0 = time.perf_counter()
ae = train(beta=0, plain_autoencoder=True)
with torch.no_grad():
    mu_ae, _ = ae.encode(X_test_t.to(device))
print(f"AE: {time.perf_counter() - t0:.0f}s | latent của ảnh thật: trung bình theo chiều ∈ [{mu_ae.mean(0).min():.1f}, {mu_ae.mean(0).max():.1f}], "
      f"độ lệch chuẩn ∈ [{mu_ae.std(0).min():.1f}, {mu_ae.std(0).max():.1f}]")
show_grid(ae.sample(12, torch.Generator().manual_seed(0)), "AE: giải mã z ~ N(0, I)")

Latent của autoencoder có độ lớn tuỳ ý (thang khác hẳn $\mathcal N(0, I)$) → giải mã điểm ngẫu nhiên cho ra "nhiễu xám", không phải hình.

## 2. Variational Autoencoder

VAE (Kingma & Welling, 2014) xem dữ liệu sinh ra từ một biến ẩn: $z \sim p(z) = \mathcal N(0, I)$, rồi $x \sim p_\theta(x\mid z)$ (decoder). Encoder không trả một điểm mà một **phân phối** $q_\phi(z\mid x) = \mathcal N(\mu(x), \text{diag}\,\sigma^2(x))$. Cực đại cận dưới của log-likelihood (**ELBO**):

$$\log p(x) \ge \underbrace{\mathbb E_{q(z|x)}[\log p(x\mid z)]}_{\text{tái tạo}} - \underbrace{\text{KL}\big(q(z\mid x)\,\|\,\mathcal N(0, I)\big)}_{\text{kéo latent về prior}}$$

- Số hạng **KL** ép mã của mọi ảnh nằm quanh $\mathcal N(0, I)$ và có độ "nhoè" $\sigma$ → không gian latent **liền mạch**, lấp đầy → giải mã điểm ngẫu nhiên từ prior ra ảnh hợp lý.
- **Reparameterization trick**: $z = \mu + \sigma\odot\varepsilon$, $\varepsilon\sim\mathcal N(0, I)$ — tính ngẫu nhiên tách khỏi tham số nên gradient đi qua $\mu, \sigma$ được.
- KL giữa hai Gaussian có công thức đóng: $-\frac12\sum_j\left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$.

In [ ]:
t0 = time.perf_counter()
vae = train(beta=1.0)
with torch.no_grad():
    logits, mu, logvar = vae(X_test_t.to(device))
    recon, kl = elbo_terms(logits, X_test_t.to(device), mu, logvar)
print(f"VAE (β = 1): {time.perf_counter() - t0:.0f}s | tái tạo = {recon:.1f} nats/ảnh | KL = {kl:.1f} nats | −ELBO = {recon + kl:.1f}")
print(f"latent của ảnh thật: trung bình ∈ [{mu.mean(0).min():.2f}, {mu.mean(0).max():.2f}], độ lệch chuẩn ∈ [{mu.std(0).min():.2f}, {mu.std(0).max():.2f}]")
show_grid(vae.sample(12, torch.Generator().manual_seed(0)), "VAE: giải mã z ~ N(0, I)")

Mẫu từ prior giờ là các **hình** (dù hơi mờ, đôi khi méo — VAE nhỏ, ảnh nhỏ). Latent của ảnh thật có trung bình gần 0 và độ lệch chuẩn cỡ 1 — đúng thang của prior $\mathcal N(0, I)$, khác hẳn autoencoder ở trên. Khi KL được nhấn mạnh hơn (mục 3), một số chiều latent còn bị đẩy hẳn về prior và decoder bỏ qua chúng — VAE tự chọn số chiều cần dùng.

### Nội suy trong không gian latent

In [ ]:
with torch.no_grad():
    (mu_a, _), (mu_b, _) = vae.encode(X_test_t[:1].to(device)), vae.encode(X_test_t[3:4].to(device))
    (ae_a, _), (ae_b, _) = ae.encode(X_test_t[:1].to(device)), ae.encode(X_test_t[3:4].to(device))
    ts = torch.linspace(0, 1, 12, device=device)[:, None]
    path_vae = torch.sigmoid(vae.decoder(mu_a + ts * (mu_b - mu_a))).cpu()
    path_ae = torch.sigmoid(ae.decoder(ae_a + ts * (ae_b - ae_a))).cpu()
show_grid(path_vae, f"VAE: nội suy {SHAPE_CLASSES[y_test[0]]} → {SHAPE_CLASSES[y_test[3]]}")
show_grid(path_ae, "AE: cùng hai ảnh")

Cả hai đều chuyển dần (decoder liên tục), nhưng các điểm giữa của VAE gần với "ảnh hợp lý" hơn — vì VAE được huấn luyện để **mọi** điểm quanh mã của dữ liệu đều giải mã được (nhờ nhiễu $\sigma$ khi train).

## 3. Hệ số β: đánh đổi tái tạo ↔ cấu trúc

$\beta$-VAE (Higgins et al., 2017): loss = tái tạo + $\beta\cdot$KL. $\beta$ nhỏ → gần autoencoder: tái tạo tốt, latent kém đều đặn, mẫu sinh méo. $\beta$ lớn → latent rất gần prior (KL nhỏ, ít thông tin qua nút cổ chai) → tái tạo và mẫu **mờ**. Trong lý thuyết thông tin: KL là **rate** (số nats thông tin mã hoá về $x$), tái tạo là **distortion**.

In [ ]:
beta_models = {1.0: vae}
for beta in [0.25, 4.0]:
    beta_models[beta] = train(beta)
for beta in sorted(beta_models):
    m = beta_models[beta]
    with torch.no_grad():
        logits, mu, logvar = m(X_test_t.to(device))
        recon, kl = elbo_terms(logits, X_test_t.to(device), mu, logvar)
        mse = F.mse_loss(torch.sigmoid(m.decoder(mu)), X_test_t.to(device)).item()
    active = int((mu.std(0) > 0.1).sum())
    print(f"β = {beta:4.2f}: KL (rate) = {kl:5.1f} nats | tái tạo = {recon:6.1f} nats | MSE tái tạo = {mse:.4f} | số chiều latent hoạt động = {active}/{K}")
for beta in sorted(beta_models):
    show_grid(beta_models[beta].sample(12, torch.Generator().manual_seed(0)), f"β = {beta}: mẫu từ prior")

Tăng $\beta$: KL giảm, sai số tái tạo tăng, ít chiều latent được dùng hơn (với $\beta = 4$, phần lớn các chiều "chết": độ lệch chuẩn của $\mu$ gần 0), mẫu mờ và đơn điệu dần; giảm $\beta$: tái tạo sắc hơn nhưng mẫu từ prior nhiều hình méo (vùng latent giữa các ảnh thật chưa được "lấp"). Không có $\beta$ "đúng" — tuỳ mục tiêu (nén, sinh, biểu diễn tách rời).

## 4. Đánh giá mô hình sinh: khó hơn ta tưởng

Không có "đáp án" để so mẫu sinh. Một ý tưởng phổ biến: dùng một **bộ phân loại** đã train trên ảnh thật — mẫu tốt thì bộ phân loại nhận ra tự tin, mẫu đa dạng thì phân bố lớp đều (tinh thần của Inception Score). Thử với chính các mẫu ở trên:

In [ ]:
torch.manual_seed(0)
clf = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(32 * 7 * 7, 5)).to(device)
opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
y_train_t = torch.tensor(y_train)
for _ in range(1500):
    idx = torch.randint(0, len(X_train_t), (128,), generator=g)
    opt.zero_grad()
    F.cross_entropy(clf(X_train_t[idx].to(device)), y_train_t[idx].to(device)).backward()
    opt.step()
clf.eval()
with torch.no_grad():
    print(f"bộ phân loại trên ảnh thật: accuracy = {(clf(X_test_t.to(device)).argmax(1).cpu() == torch.tensor(y_test)).float().mean():.3f}")
    for name, m in [("AE (nhiễu xám)", ae), ("VAE β = 1", vae), ("VAE β = 4", beta_models[4.0])]:
        p = torch.softmax(clf(m.sample(2000, torch.Generator().manual_seed(1)).to(device)), 1).cpu()
        freq = torch.bincount(p.argmax(1), minlength=5).float() / len(p)
        print(f"{name:15s}: tỉ lệ mẫu được nhận ra với xác suất > 0.9 = {(p.max(1).values > 0.9).float().mean():.2f} | "
              f"phân bố lớp dự đoán = {np.round(freq.numpy(), 2)}")

Bộ phân loại "tự tin" cả với **nhiễu xám** của autoencoder — mạng phân loại thường quá tự tin với dữ liệu ngoài phân phối (chương 37). Chỉ số dựa trên độ tự tin của bộ phân loại vì thế dễ bị đánh lừa; phân bố lớp cho thấy thêm vấn đề **đa dạng** (mẫu dồn vào vài lớp dễ). Thực hành đánh giá mô hình sinh: **FID** (so phân phối đặc trưng của ảnh thật và ảnh sinh trong không gian của một mạng tiền huấn luyện), precision/recall cho mô hình sinh (chất lượng vs độ phủ), kiểm tra **ghi nhớ** (mẫu sinh có trùng ảnh train không — láng giềng gần nhất), và **đánh giá của con người**.

## 5. Bản đồ các mô hình sinh

| Họ | Ý tưởng | Ưu | Nhược |
|---|---|---|---|
| VAE | encoder xác suất + decoder, tối ưu ELBO | huấn luyện ổn định, có encoder, latent liền mạch | mẫu thường mờ |
| GAN | generator đánh lừa discriminator (trò chơi đối kháng) | ảnh sắc nét | huấn luyện bất ổn, mode collapse, không có encoder |
| Autoregressive | sinh từng phần tử theo xác suất có điều kiện (GPT — chương 31) | likelihood chính xác, rất mạnh với văn bản | sinh tuần tự, chậm |
| Normalizing flow | biến đổi khả nghịch từ Gaussian | likelihood chính xác | kiến trúc bị ràng buộc |
| Diffusion | học **khử nhiễu** từng bước (≈ denoising AE nhiều mức nhiễu — bài 01), sinh bằng cách khử nhiễu dần từ nhiễu thuần | chất lượng và độ đa dạng hàng đầu (Stable Diffusion, DALL·E) | sinh nhiều bước, tốn tính toán |

Latent diffusion (Stable Diffusion) **kết hợp** hai ý tưởng: một autoencoder (kiểu VAE) nén ảnh vào latent, rồi mô hình diffusion chạy trong không gian latent đó.

## 🧠 Kiểm tra nhanh

1. Vì sao giải mã một điểm latent ngẫu nhiên của autoencoder thường ra ảnh vô nghĩa?
2. Viết ELBO. Mỗi số hạng làm gì?
3. Reparameterization trick giải quyết vấn đề gì?
4. β lớn và β nhỏ ảnh hưởng thế nào tới tái tạo, latent và mẫu sinh?
5. Vì sao độ tự tin của một bộ phân loại không đủ để đánh giá mẫu sinh?

## 📌 Tóm tắt

- VAE = autoencoder xác suất: encoder cho $\mathcal N(\mu, \sigma^2)$, decoder $p(x\mid z)$, tối ưu ELBO = tái tạo − KL với prior $\mathcal N(0, I)$.
- KL làm latent liền mạch → lấy mẫu từ prior và nội suy được; reparameterization cho phép lan truyền ngược qua phép lấy mẫu.
- β điều chỉnh rate–distortion; đánh giá mô hình sinh cần nhiều góc nhìn (chất lượng, đa dạng, ghi nhớ).
- GAN, autoregressive, flow, diffusion là các họ khác; diffusion ≈ khử nhiễu nhiều bước.

## 📚 Tài liệu tham khảo

- Kingma & Welling (2014) — *Auto-Encoding Variational Bayes*; Kingma & Welling (2019) — *An Introduction to Variational Autoencoders*
- Higgins et al. (2017) — *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*
- Goodfellow et al. (2014) — *Generative Adversarial Nets*
- Ho, Jain & Abbeel (2020) — *Denoising Diffusion Probabilistic Models*; Rombach et al. (2022) — *High-Resolution Image Synthesis with Latent Diffusion Models*
- Heusel et al. (2017) — *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium* (FID)